In [1]:
import torch
import torch.nn as nn
from transformers import RobertaModel

class MultiTaskRoberta(nn.Module):
    def __init__(self):
        super().__init__()
        self.roberta = RobertaModel.from_pretrained('roberta-base')
        self.dropout = nn.Dropout(0.1)
        # Two separate heads
        self.binary_head = nn.Linear(768, 2)       # binary classification
        self.severity_head = nn.Linear(768, 1)      # regression 0-4

    def forward(self, input_ids, attention_mask, labels=None, severity_labels=None):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.dropout(outputs.pooler_output)

        binary_logits = self.binary_head(pooled)
        severity_pred = self.severity_head(pooled).squeeze(-1)

        loss = None
        if labels is not None and severity_labels is not None:
            alpha = 0.7
            binary_loss = nn.CrossEntropyLoss()(binary_logits, labels)
            severity_loss = nn.MSELoss()(severity_pred, severity_labels)
            loss = alpha * binary_loss + (1 - alpha) * severity_loss

        return {'loss': loss, 'logits': binary_logits}


class OrdinalRoberta(nn.Module):
    def __init__(self):
        super().__init__()
        self.roberta = RobertaModel.from_pretrained('roberta-base')
        self.dropout = nn.Dropout(0.1)
        self.ordinal_head = nn.Linear(768, 4)  # P(>=1), P(>=2), P(>=3), P(>=4)

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.dropout(outputs.pooler_output)
        logits = self.ordinal_head(pooled)

        loss = None
        if labels is not None:
            # Convert orig_label to ordinal targets
            # e.g. orig_label=2 -> [1, 1, 0, 0]
            targets = torch.zeros_like(logits)
            for i in range(4):
                targets[:, i] = (labels > i).float()
            loss = nn.BCEWithLogitsLoss()(logits, targets)

        return {'loss': loss, 'logits': logits}

In [12]:
# inference.py
import sys
import os

import torch
import numpy as np
import pandas as pd
from scipy.special import softmax
from transformers import AutoTokenizer, RobertaForSequenceClassification, AutoModelForSequenceClassification
from huggingface_hub import hf_hub_download
import json

# Load config
with open('ensemble_config.json') as f:
    config = json.load(f)
weights = config['weights']

# Ensure 'weighted' key exists in weights and re-normalize if necessary
if 'weighted' not in weights:
    weights['weighted'] = 0.1 # Assign a default weight if missing
    # Recalculate total_weight and normalize all weights
    total_weight = sum(weights.values())
    weights = {k: v / total_weight for k, v in weights.items()}

threshold = config['threshold']

# Determine device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load tokenizers
roberta_tokenizer = AutoTokenizer.from_pretrained('brandonwooding/patronize-baseline-roberta')
deberta_tokenizer = AutoTokenizer.from_pretrained('brandonwooding/patronize-baseline-deberta')

# Load HuggingFace models and move to device
baseline_model = RobertaForSequenceClassification.from_pretrained('brandonwooding/patronize-baseline-roberta').to(device)
weighted_model = RobertaForSequenceClassification.from_pretrained('brandonwooding/patronize-weighted-roberta').to(device)
soft_model = RobertaForSequenceClassification.from_pretrained('brandonwooding/patronize-soft-roberta').to(device)
deberta_model = AutoModelForSequenceClassification.from_pretrained('brandonwooding/patronize-baseline-deberta').to(device)

# Load custom models from HuggingFace and move to device
multi_path = hf_hub_download(repo_id='brandonwooding/patronize-multitask-roberta', filename='multi_task_roberta.pt')
ordinal_path = hf_hub_download(repo_id='brandonwooding/patronize-ordinal-roberta', filename='ordinal_roberta.pt')

multi_model = MultiTaskRoberta().to(device)
multi_model.load_state_dict(torch.load(multi_path, map_location=device))
multi_model.eval()

ordinal_model = OrdinalRoberta().to(device)
ordinal_model.load_state_dict(torch.load(ordinal_path, map_location=device))
ordinal_model.eval()

# Set all models to eval mode
baseline_model.eval()
weighted_model.eval()
soft_model.eval()
deberta_model.eval()


def predict_ensemble(df, batch_size=32):
    """Takes a dataframe with 'keyword' and 'paragraph' columns, returns predictions and probabilities."""

    all_preds = []
    all_weighted_prob = []

    for i in range(0, len(df), batch_size):
        batch_df = df.iloc[i:i + batch_size]
        texts = (batch_df['keyword'].fillna('').astype(str) + ' : ' + batch_df['paragraph'].fillna('').astype(str)).tolist()

        # Tokenize and move to device
        rob_enc = roberta_tokenizer(texts, truncation=True, padding='max_length', max_length=256, return_tensors='pt')
        rob_enc = {k: v.to(device) for k, v in rob_enc.items()}

        deb_enc = deberta_tokenizer(texts, truncation=True, padding='max_length', max_length=256, return_tensors='pt')
        deb_enc = {k: v.to(device) for k, v in deb_enc.items()}

        with torch.no_grad():
            # Standard models
            prob_baseline = softmax(baseline_model(**rob_enc).logits.cpu().numpy(), axis=1)[:, 1]
            prob_weighted = softmax(weighted_model(**rob_enc).logits.cpu().numpy(), axis=1)[:, 1]
            prob_soft = softmax(soft_model(**rob_enc).logits.cpu().numpy(), axis=1)[:, 1]
            prob_deberta = softmax(deberta_model(**deb_enc).logits.cpu().numpy(), axis=1)[:, 1]

            # Custom models
            multi_out = multi_model(input_ids=rob_enc['input_ids'], attention_mask=rob_enc['attention_mask'])
            prob_multi = softmax(multi_out['logits'].cpu().numpy(), axis=1)[:, 1]

            ordinal_out = ordinal_model(input_ids=rob_enc['input_ids'], attention_mask=rob_enc['attention_mask'])
            prob_ordinal = torch.sigmoid(ordinal_out['logits'][:, 1]).cpu().numpy()

        # Weighted average
        weighted_prob = (
            weights['baseline'] * prob_baseline +
            weights['weighted'] * prob_weighted +
            weights['soft'] * prob_soft +
            weights['multi'] * prob_multi +
            weights['ordinal'] * prob_ordinal +
            weights['deberta'] * prob_deberta
        )

        preds = (weighted_prob > threshold).astype(int)

        all_preds.extend(preds)
        all_weighted_prob.extend(weighted_prob)

    return np.array(all_preds), np.array(all_weighted_prob)


def save_predictions(preds, filename):
    """Save predictions to txt file, one per line."""
    with open(filename, 'w') as f:
        for pred in preds:
            f.write(f"{pred}\n")

Using device: cuda


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [14]:
# Load your data
dev_df = pd.read_csv('dev.csv')
test_df = pd.read_csv('test.csv')

# Get predictions
dev_preds, dev_probs = predict_ensemble(dev_df)
test_preds, test_probs = predict_ensemble(test_df)

# Save to txt
save_predictions(dev_preds, 'results/dev.txt')
save_predictions(test_preds, 'results/test.txt')

# If dev set has labels, print F1
if 'label' in dev_df.columns:
    from sklearn.metrics import f1_score
    print(f"Dev F1: {f1_score(dev_df['label'].values, dev_preds):.4f}")

print(f"Dev predictions saved: {len(dev_preds)}")
print(f"Test predictions saved: {len(test_preds)}")

Dev F1: 0.6364
Dev predictions saved: 2094
Test predictions saved: 3832
